## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
import geopandas as gpd

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [3]:
from gridsample.utils import create_ids, save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
CLEANED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "01 Cleaned Khasras"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "03 Suggested Parcels"

## Load cleaned khasras

In [5]:
# # Dhar
# dhar_processed_areas_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "dhar_cleaned_khasras.parquet")
# dhar_processed_areas_gdf["source"] = "dhar"

In [6]:
# Sagar
sagar_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "sagar_cleaned_khasras.parquet")
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

In [ ]:
sagar_gdf

## Combine khasras into continuous parcels

In [8]:
def build_graph_from_gdf(gdf):
    """
    Build a graph from a GeoDataFrame where the nodes are the index of the GeoDataFrame
    and the edges are the distance between the geometries.

    Parameters
    ----------
    gdf : GeoDataFrame
        The GeoDataFrame to build the graph from.

    Returns
    -------
    nx.Graph
        The graph with the distances as the edge attribute.
    """

    gdf_index = list(gdf.index)
    G = nx.Graph()
    for i, geom1 in enumerate(gdf.geometry):
        for j, geom2 in enumerate(gdf.geometry):
            if i <= j:
                distance = geom1.distance(geom2)
                G.add_edge(gdf_index[i], gdf_index[j], distance=distance)

    print(
        f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges."
    )
    return G

In [9]:
def filter_graph_by_distance_threshold(G, threshold):
    """Filter a graph by a distance threshold."""

    G_filtered = G.copy()
    for edge in G.edges(data=True):
        if edge[2]["distance"] > threshold:
            G_filtered.remove_edge(edge[0], edge[1])
    return G_filtered

In [10]:
def get_connected_components_by_distance_threshold(G, distance_threshold):
    """
    Get the connected components of a graph by a distance threshold. The connected components are
    the nodes that are within the distance threshold of each other.

    Parameters
    ----------
    G : nx.Graph
        The graph to get the connected components from.
    distance_threshold : float
        The distance threshold to use to get the connected components.

    Returns
    -------
    cluster_labels_df : pd.DataFrame
        A DataFrame that maps each node (index) to its cluster label.
    G_with_cluster_labels : nx.Graph
        The graph with the cluster labels as node attributes.
    """

    # filter to only the edges that are within the distance threshold and get the connected components
    G_filtered = filter_graph_by_distance_threshold(G, distance_threshold)
    # print(len(G.edges), "edges filtered to", len(G_filtered.edges), "by distance threshold", distance_threshold)
    list_of_sets_of_connected_nodes = list(nx.connected_components(G_filtered))

    # create parcel ids
    parcel_ids = create_ids(len(list_of_sets_of_connected_nodes), prefix="PARCEL_")

    # create a dictionary that maps each node to its parcel_id.
    node_to_parcel_id = {}
    for parcel_id, connected_nodes in zip(parcel_ids, list_of_sets_of_connected_nodes):
        for node in connected_nodes:
            node_to_parcel_id[node] = parcel_id

    # create a dataframe that maps each node (as index) to its parcel_id
    cluster_labels_df = pd.DataFrame(
        {"parcel_id": node_to_parcel_id.values()},
        index=node_to_parcel_id.keys(),
    )

    # add the parcel_id attribute to the nodes
    G_with_parcel_id = G.copy()
    for node in G_with_parcel_id.nodes:
        G_with_parcel_id.nodes[node]["parcel_id"] = node_to_parcel_id[node]

    return cluster_labels_df, G_with_parcel_id

In [11]:
gdf = sagar_gdf.to_crs("24378").reset_index(drop=True)[
    [
        "geometry",
        "UNQID",
        "village_name",
    ]
]

In [ ]:
gdf.plot(column="UNQID")

In [ ]:
# get graph
G = build_graph_from_gdf(gdf)

In [14]:
# get grouping
distance_threshold = 25
cluster_labels_df, G_with_parcel_id = get_connected_components_by_distance_threshold(
    G, distance_threshold
)

In [15]:
# add parcel_id to gdf
gdf_with_parcel_id = gdf.merge(cluster_labels_df, left_index=True, right_index=True)

In [16]:
gdf_with_parcel_id["Khasra Area (m2)"] = gdf_with_parcel_id.area

In [17]:
# rename any cluster that has only 1 khasra and that khasra is less than 100 hectares to DISCARDED
gdf_with_parcel_id.loc[
    (
        gdf_with_parcel_id["parcel_id"].map(
            gdf_with_parcel_id["parcel_id"].value_counts()
        )
        == 1
    )
    & (gdf_with_parcel_id["Khasra Area (m2)"] < 1000000),
    "parcel_id",
] = "DISCARDED"

In [19]:
save_shapefiles(
    gdf_with_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / "Sagar",
    "sagar_khasras_with_parcel_id",
    formats=["parquet", "kml", "csv"],
)

In [ ]:
gdf_with_parcel_id.plot(column="parcel_id", cmap="jet", figsize=(8, 8), legend=True)

In [353]:
# group by the cluster value and dissolve to make a new GeoDataFrame
clustered_gdf = gdf_with_parcel_id.dissolve(by="parcel_id")
clustered_gdf = clustered_gdf.drop(columns=["UNQID", "Khasra Area (m2)"])

In [354]:
# add the names of all khasras that fall inside each parcel as a list under khasra_ids
clustered_gdf.loc[:, "khasra_ids"] = (
    gdf_with_parcel_id.groupby("parcel_id")["UNQID"].apply(list).astype(str)
)

In [355]:
clustered_gdf = clustered_gdf.reset_index()

In [356]:
# drop the discard cluster
clustered_gdf = clustered_gdf[clustered_gdf["parcel_id"] != "DISCARDED"]

In [357]:
clustered_gdf.loc[:, "Parcel Area (m2)"] = clustered_gdf["geometry"].area

In [358]:
larger_than_100ha = clustered_gdf[clustered_gdf["Parcel Area (m2)"] > 1000000]

In [ ]:
larger_than_100ha.plot(column="Parcel Area (m2)", legend=True)

In [361]:
def calculate_min_distance_and_closest_id(gdf):
    min_distances = []
    closest_ids = []
    for i in range(len(gdf)):
        geom = gdf.iloc[i].geometry
        other_geoms = gdf.drop(gdf.index[i])
        distances = other_geoms.geometry.apply(lambda x: geom.distance(x))
        min_distance = distances.min().round(2)
        closest_id = other_geoms.loc[distances.idxmin()]["parcel_id"]
        min_distances.append(min_distance)
        closest_ids.append(closest_id)
    return min_distances, closest_ids

In [ ]:
# Calculate minimum distances and closest parcel_ids
min_distances, closest_ids = calculate_min_distance_and_closest_id(larger_than_100ha)

# Add the results as new columns
larger_than_100ha.loc[:, "Distance to Closest Parcel (m)"] = min_distances
larger_than_100ha.loc[:, "ID of Closest Parcel"] = closest_ids

In [364]:
# make KMZ with khasras but coloured in + layer for parcel shapes

## Save files

In [366]:
save_shapefiles(
    larger_than_100ha.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / "Sagar",
    "sagar_parcels_initial",
    formats=["parquet", "kml", "csv"],
)

## Scraps

In [ ]:
# def calculate_distance_matrix_old(gdf):
#     distances = gdf.geometry.apply(lambda geom1: gdf.geometry.distance(geom1))
#     return distances.to_numpy()

In [ ]:
# def calculate_distance_matrix_upper(gdf):
#     n = len(gdf)
#     distance_matrix_upper = np.full((n, n), np.nan)
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i < j:
#                 distance = geom1.distance(geom2)
#                 distance_matrix_upper[i, j] = distance
#                 distance_matrix_upper[j, i] = distance
#             if i == j:
#                 distance_matrix_upper[i, j] = 0
#     return distance_matrix_upper

In [ ]:
# def cluster_adjacent_shapes_old(gdf, distance_threshold):
#     gdf_index = gdf.index

#     # Step 2: Create a distance-filtered spatial graph
#     G = nx.Graph()
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i != j:
#                 distance = geom1.distance(geom2)
#                 if distance < distance_threshold:
#                     # this means far away nodes don't actually ever get added to the graph
#                     G.add_edge(i, j, distance=distance)

#     # Step 3: Find connected components
#     connected_components = list(nx.connected_components(G))

#     # Step 4: Convert the connected components to a list of labels that matches the input data
#     data = []
#     for cluster_id, value_set in enumerate(connected_components):
#         for value in value_set:
#             data.append((value, cluster_id))

#     df = pd.DataFrame(data, columns=["index", "cluster"])
#     df.set_index("index", inplace=True)

#     # make index of df the same as gdf and assign any missing values as -1
#     cluster_labels = df.reindex(gdf_index)["cluster"].fillna(-1).astype(int)

#     return cluster_labels, G, connected_components

In [22]:
# sagar_gdf[sagar_gdf["UNQID"] == "17677985"].geometry.values[0].distance(
#     sagar_gdf[sagar_gdf["UNQID"] == "17677984"].geometry.values[0]
# )

In [23]:
# sagar_gdf_4326 = sagar_gdf.to_crs("4326")
# sagar_gdf_4326["Lat"] = sagar_gdf_4326.centroid.y
# sagar_gdf_4326["Lon"] = sagar_gdf_4326.centroid.x
# create_interactive_map(sagar_gdf_4326, point_id_col="UNQID", zoom_start=12)

In [24]:
# from sklearn.cluster import HDBSCAN
# clusters = HDBSCAN(min_cluster_size=2, metric="precomputed", n_jobs=-1).fit(dist_matrix)
# gdf['clustering_dbscan'] = clusters.labels_
# gdf.plot(column='clustering_dbscan', legend=True)